# Knowledge Graph in Pure Python — `rdflib` + `networkx`

This notebook builds the **same knowledge graph as your Cypher**, but
**entirely in Python** — no Neo4j, no database server. It mirrors the schema
in `build_KG_queries.cypher` exactly, two ways side by side:

* **Part A — `rdflib`**: a true semantic KG (RDF triples + a small ontology),
  queried with **SPARQL** in Python, saved to a `.ttl` file.
* **Part B — `networkx`**: a property graph, queried with **Python / graph
  algorithms**, and rendered interactively (the Python equivalent of
  `visualize_KG_query.cypher`'s `MATCH (a)-[r]->(b)`).

### Schema (faithful to `build_KG_queries.cypher`)

| Cypher node | key | properties |
|---|---|---|
| `Case` | `caseNumber` | caseTitle, caseInstrument, caseInitiationDate, caseLastDecisionDate, decisionLabel |
| `Section` | `sectionName` | sectionCode, division, group, classCode, classDescription |
| `LegalBasis` | `name` | — (split `caseLegalBasisLabel` on `;`) |
| `Company` | `name` | — (split `caseCompanies` on `;`) |

| Cypher relationship | rdflib predicate | networkx edge key |
|---|---|---|
| `(Case)-[:HAS_SECTION]->(Section)` | `:hasSection` | `HAS_SECTION` |
| `(Case)-[:HAS_LEGAL_BASIS]->(LegalBasis)` | `:hasLegalBasis` | `HAS_LEGAL_BASIS` |
| `(Case)-[:INVOLVES_COMPANY]->(Company)` | `:involvesCompany` | `INVOLVES_COMPANY` |

In [ ]:
# Install dependencies (run once)
%pip install -q rdflib networkx pyvis pandas

## 1. Load & clean `cases.csv`

Mirrors the Cypher's `LOAD CSV` logic: `'null'`/empty -> missing, dates
truncated to the first 10 chars (`YYYY-MM-DD`), multi-value fields split on
`;` and trimmed.

In [ ]:
import csv, re
from pathlib import Path

def clean(v):
    """'' / 'null' -> None, else stripped string (mirrors Cypher null handling)."""
    if v is None:
        return None
    v = v.strip()
    return None if v == "" or v.lower() == "null" else v

def to_date(v):
    v = clean(v)
    return v[:10] if v else None          # date(substring(x,0,10))

def split_multi(v):
    v = clean(v)
    if not v:
        return []
    return [p.strip() for p in v.split(";") if p.strip()]

rows = []
with open("cases.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append({
            "caseNumber":           clean(r["caseNumber"]),
            "caseTitle":            clean(r["caseTitle"]),
            "caseInstrument":       clean(r["caseInstrument"]),
            "caseInitiationDate":   to_date(r["caseInitiationDate"]),
            "caseLastDecisionDate": to_date(r["caseLastDecisionDate"]),
            "decisionLabel":        clean(r["decisionLabel"]),
            "sectionName":          clean(r["sectionName"]),
            "sectionCode":          clean(r["sectionCode"]),
            "division":             clean(r["division"]),
            "group":                clean(r["group"]),
            "classCode":            clean(r["classCode"]),
            "classDescription":     clean(r["classDescription"]),
            "legalBases":           split_multi(r["caseLegalBasisLabel"]),
            "companies":            split_multi(r["caseCompanies"]),
        })

rows = [r for r in rows if r["caseNumber"]]
print(f"Loaded {len(rows):,} cases")
print("Example:", rows[0])

# Part A — RDF Knowledge Graph with `rdflib`

A semantic KG: every fact is a `(subject, predicate, object)` **triple**.
We declare a tiny ontology (classes + predicates) and an `EX:` namespace,
then assert one set of triples per case.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, XSD
from urllib.parse import quote

EX = Namespace("http://example.org/eucomp#")
g = Graph()
g.bind("ex", EX)
g.bind("rdfs", RDFS)

def uri(kind, key):
    """Stable URI for a node, e.g. ex:Company/DEUTSCHE%20BANK"""
    return EX[f"{kind}/{quote(str(key), safe='')}"]

# --- Ontology (classes) ---
for cls in ("Case", "Section", "LegalBasis", "Company"):
    g.add((EX[cls], RDF.type, RDFS.Class))

for case in rows:
    c = uri("Case", case["caseNumber"])
    g.add((c, RDF.type, EX.Case))
    g.add((c, EX.caseNumber, Literal(case["caseNumber"])))
    if case["caseTitle"]:
        g.add((c, RDFS.label, Literal(case["caseTitle"])))
        g.add((c, EX.caseTitle, Literal(case["caseTitle"])))
    if case["caseInstrument"]:
        g.add((c, EX.caseInstrument, Literal(case["caseInstrument"])))
    if case["decisionLabel"]:
        g.add((c, EX.decisionLabel, Literal(case["decisionLabel"])))
    if case["caseInitiationDate"]:
        g.add((c, EX.caseInitiationDate, Literal(case["caseInitiationDate"], datatype=XSD.date)))
    if case["caseLastDecisionDate"]:
        g.add((c, EX.caseLastDecisionDate, Literal(case["caseLastDecisionDate"], datatype=XSD.date)))

    # (Case)-[:HAS_SECTION]->(Section)
    if case["sectionName"]:
        s = uri("Section", case["sectionName"])
        g.add((s, RDF.type, EX.Section))
        g.add((s, RDFS.label, Literal(case["sectionName"])))
        for prop in ("sectionCode", "division", "group", "classCode", "classDescription"):
            if case[prop]:
                g.add((s, EX[prop], Literal(case[prop])))
        g.add((c, EX.hasSection, s))

    # (Case)-[:HAS_LEGAL_BASIS]->(LegalBasis)
    for lb in case["legalBases"]:
        l = uri("LegalBasis", lb)
        g.add((l, RDF.type, EX.LegalBasis))
        g.add((l, RDFS.label, Literal(lb)))
        g.add((c, EX.hasLegalBasis, l))

    # (Case)-[:INVOLVES_COMPANY]->(Company)
    for comp in case["companies"]:
        co = uri("Company", comp)
        g.add((co, RDF.type, EX.Company))
        g.add((co, RDFS.label, Literal(comp)))
        g.add((c, EX.involvesCompany, co))

print(f"RDF graph built: {len(g):,} triples")

### Save the graph to a file (Turtle)

This `.ttl` file *is* the knowledge graph on disk — the Python-native
equivalent of having it in a database. It can be reloaded or shared.

In [ ]:
g.serialize(destination="knowledge_graph.ttl", format="turtle")
print("Wrote knowledge_graph.ttl")
print("--- first 1500 chars ---")
print(Path("knowledge_graph.ttl").read_text(encoding="utf-8")[:1500])

### Query the RDF graph with SPARQL

SPARQL is the standard query language for RDF — the semantic-web counterpart
of Cypher. These run fully in Python against the in-memory graph.

In [ ]:
import pandas as pd

def sparql(query):
    return pd.DataFrame(g.query(query).bindings)

# How many nodes of each class?
display(sparql("""
    PREFIX ex: <http://example.org/eucomp#>
    SELECT ?class (COUNT(?s) AS ?n) WHERE {
        ?s a ?class . FILTER(?class IN (ex:Case, ex:Section, ex:LegalBasis, ex:Company))
    } GROUP BY ?class ORDER BY DESC(?n)
"""))

# Companies involved in the most cases
display(sparql("""
    PREFIX ex: <http://example.org/eucomp#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?company (COUNT(?c) AS ?cases) WHERE {
        ?c ex:involvesCompany ?co . ?co rdfs:label ?company .
    } GROUP BY ?company ORDER BY DESC(?cases) LIMIT 10
"""))

# Cases per economic section
display(sparql("""
    PREFIX ex: <http://example.org/eucomp#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?section (COUNT(?c) AS ?cases) WHERE {
        ?c ex:hasSection ?s . ?s rdfs:label ?section .
    } GROUP BY ?section ORDER BY DESC(?cases) LIMIT 10
"""))

In [ ]:
# Drill into one case: every triple where it is the subject
display(sparql("""
    PREFIX ex: <http://example.org/eucomp#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?p ?label WHERE {
        ex:Case/2027 ?p ?o .
        OPTIONAL { ?o rdfs:label ?label }
    }
"""))

# Part B — Property Graph with `networkx`

Same data, modelled as a directed multigraph. Every node carries a
`node_type`; every edge a relationship type — so you can run **graph
algorithms** (degree, centrality, components) in pure Python.

In [ ]:
import networkx as nx

NG = nx.MultiDiGraph()

def nid(kind, key):                       # unique node id
    return f"{kind}:{key}"

for case in rows:
    c = nid("Case", case["caseNumber"])
    NG.add_node(c, node_type="Case", label=case["caseTitle"] or case["caseNumber"],
                instrument=case["caseInstrument"], decisionLabel=case["decisionLabel"],
                initiationDate=case["caseInitiationDate"])

    if case["sectionName"]:
        s = nid("Section", case["sectionName"])
        NG.add_node(s, node_type="Section", label=case["sectionName"],
                    sectionCode=case["sectionCode"])
        NG.add_edge(c, s, key="HAS_SECTION")

    for lb in case["legalBases"]:
        l = nid("LegalBasis", lb)
        NG.add_node(l, node_type="LegalBasis", label=lb)
        NG.add_edge(c, l, key="HAS_LEGAL_BASIS")

    for comp in case["companies"]:
        co = nid("Company", comp)
        NG.add_node(co, node_type="Company", label=comp)
        NG.add_edge(c, co, key="INVOLVES_COMPANY")

from collections import Counter
print(f"networkx graph: {NG.number_of_nodes():,} nodes, {NG.number_of_edges():,} edges")
print("Nodes by type:", Counter(d["node_type"] for _, d in NG.nodes(data=True)))
print("Edges by type:", Counter(k for _, _, k in NG.edges(keys=True)))

### Graph analytics (the payoff of a property graph)

In [ ]:
import pandas as pd

def top_by_degree(node_type, n=10):
    items = [(NG.nodes[x]["label"], NG.degree(x))
             for x in NG if NG.nodes[x]["node_type"] == node_type]
    return pd.DataFrame(sorted(items, key=lambda t: -t[1])[:n],
                        columns=[node_type, "degree (connections)"])

print("Most-connected companies:");  display(top_by_degree("Company"))
print("Busiest sections:");          display(top_by_degree("Section"))
print("Most-used legal bases:");     display(top_by_degree("LegalBasis"))

# Companies that co-appear in cases (project the bipartite graph)
und = NG.to_undirected()
pairs = Counter()
for case in rows:
    cos = sorted(set(case["companies"]))
    for i in range(len(cos)):
        for j in range(i + 1, len(cos)):
            pairs[(cos[i], cos[j])] += 1
print("Top company co-occurrences:")
display(pd.DataFrame([(a, b, n) for (a, b), n in pairs.most_common(10)],
                     columns=["company_a", "company_b", "shared_cases"]))

### Visualize — Python equivalent of `visualize_KG_query.cypher`

`MATCH (a)-[r]->(b) RETURN a, r, b` returns the whole graph. Rendering all
~11k cases in a browser is unreadable, so we sample `SAMPLE_CASES` cases and
all their neighbours into an interactive `knowledge_graph.html`
(zoom / drag / hover). Raise `SAMPLE_CASES` for more, or set it to `None`
for the full graph (heavy).

In [ ]:
from pyvis.network import Network

SAMPLE_CASES = 120          # None = entire graph (slow / huge HTML)

COLORS = {"Case": "#4C9BE8", "Company": "#E8804C",
          "Section": "#56C271", "LegalBasis": "#B07CE8"}

if SAMPLE_CASES is None:
    sub = NG
else:
    keep = set()
    for case in rows[:SAMPLE_CASES]:
        c = nid("Case", case["caseNumber"]); keep.add(c)
        keep.update(NG.successors(c))
    sub = NG.subgraph(keep)

net = Network(height="750px", width="100%", directed=True,
              bgcolor="#ffffff", notebook=False)
for n, d in sub.nodes(data=True):
    net.add_node(n, label=d["label"][:28], title=f'{d["node_type"]}: {d["label"]}',
                 color=COLORS.get(d["node_type"], "#999"),
                 size=12 + (3 if d["node_type"] == "Case" else 0))
for a, b, k in sub.edges(keys=True):
    net.add_edge(a, b, title=k, label=k)
net.repulsion(node_distance=170, spring_length=140)
net.write_html("knowledge_graph.html", open_browser=False, notebook=False)
print(f"Wrote knowledge_graph.html  ({sub.number_of_nodes()} nodes, "
      f"{sub.number_of_edges()} edges) — open it in a browser.")

In [ ]:
# Static snapshot of ONE case's neighbourhood (no browser needed)
import matplotlib.pyplot as plt

center = nid("Case", rows[0]["caseNumber"])
ego = nx.ego_graph(NG, center, radius=1).to_undirected()
pos = nx.spring_layout(ego, seed=42, k=0.9)
plt.figure(figsize=(11, 7))
for nt, col in COLORS.items():
    ns = [n for n in ego if NG.nodes[n]["node_type"] == nt]
    nx.draw_networkx_nodes(ego, pos, nodelist=ns, node_color=col,
                           node_size=900, label=nt)
nx.draw_networkx_edges(ego, pos, alpha=0.4)
nx.draw_networkx_labels(ego, pos,
    labels={n: NG.nodes[n]["label"][:18] for n in ego}, font_size=8)
plt.legend(scatterpoints=1); plt.title(f"Ego graph of {rows[0]['caseTitle']}")
plt.axis("off"); plt.tight_layout(); plt.show()

## Summary — how this maps back to your Cypher

| Your Cypher | This notebook (Python) |
|---|---|
| `LOAD CSV` + `MERGE` nodes | Part A `rdflib` triples / Part B `networkx` nodes |
| `HAS_SECTION` / `HAS_LEGAL_BASIS` / `INVOLVES_COMPANY` | same predicates / edge types |
| `MATCH ... RETURN` analytics | **SPARQL** (Part A) or **Python algorithms** (Part B) |
| `visualize_KG_query.cypher` | interactive `knowledge_graph.html` |
| graph stored in Neo4j | `knowledge_graph.ttl` (RDF) + in-memory `networkx` |

Both parts are pure Python — they run with no database. Use **rdflib** when
you need formal semantics / SPARQL / RDF export; use **networkx** when you
need graph algorithms and fast visual exploration.